In [ ]:
# ==============================================================================
# STEP 1: SETUP & IMPORTS
# ==============================================================================

# 1. INSTALLATION
# Installing all dependencies. Diffusers already includes the new schedulers.
!pip install -q encodec diffusers transformers accelerate soundfile librosa

# 2. SYSTEM & BASICS
import os
import glob
import json
import time
import warnings
import numpy as np
from tqdm.auto import tqdm
from google.colab import drive # Drive import here

# 3. PYTORCH & DEEP LEARNING
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset

# 4. HUGGING FACE STACK (Diffusers & Transformers)
# --- THIS IS THE IMPORTANT CHANGE ---
# Importing DPMSolverMultistepScheduler for fast inference
# and DDIMScheduler as an alternative. DDPMScheduler remains for reference.
from diffusers import (
    UNet2DConditionModel,
    DDPMScheduler,
    DDIMScheduler,
    DPMSolverMultistepScheduler
)
from accelerate import Accelerator
from transformers import T5Tokenizer, T5EncoderModel

# 5. AUDIO & SIGNAL PROCESSING
import torchaudio
import soundfile as sf
import librosa
import librosa.display
from encodec import EncodecModel

# 6. VISUALIZATION & OUTPUT
import matplotlib.pyplot as plt
import IPython.display as ipd

# 7. HARDWARE INITIALIZATION
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
warnings.filterwarnings("ignore")

# 8. MOUNT GOOGLE DRIVE
# Mounted here to allow saving/loading results later
drive.mount('/content/drive')

print(f"✅ All libraries loaded (including DPMSolver++ for MIMII-Gen).")
print(f"🚀 Hardware check: {device}")
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"📊 Available VRAM: {vram:.2f} GB")

In [ ]:
# ==============================================================================
# SANITY CHECK: GROUND TRUTH WITH ENERGY ANALYSIS
# ==============================================================================
import torch
import librosa
import librosa.display
import numpy as np
import IPython.display as ipd
import matplotlib.pyplot as plt
import scipy.signal as signal
from encodec import EncodecModel

# 1. Setup
if 'encodec_model' not in locals():
    print("⏳ Loading EnCodec...")
    encodec_model = EncodecModel.encodec_model_24khz()
    encodec_model.to(device)
    encodec_model.set_target_bandwidth(24.0)
    encodec_model.eval()

# 2. Load a REAL file (Ground Truth)
# Path to an existing .pt file
real_file_path = "/content/drive/MyDrive/MasterProject/data/features/encodec/train/bearing/section_00_source_train_normal_0000_vel_22.pt"

print(f"🧪 CHECKING DECODING PIPELINE WITH REAL FILE...")
print(f"📂 Loading: {real_file_path}")

try:
    # A. Loading
    real_latent = torch.load(real_file_path, map_location=device)
    if real_latent.dim() == 3:
        real_latent = real_latent.unsqueeze(0)

    # B. Stats & Pipeline Simulation
    # Normalizing to simulate the model input, then denormalizing immediately.

    # 1. Reshape for stats [1, 16, 8, T]
    b, c, h, w = real_latent.shape
    lat_for_stats = real_latent.view(b, 16, 8, w)

    # 2. Normalize (Simulation)
    lat_norm = (lat_for_stats - GLOBAL_MEAN) / (GLOBAL_STD + 1e-6)

    print(f"    Stats Check -> Mean: {lat_norm.mean():.4f} (Target ~0)")
    print(f"    Stats Check -> Std:  {lat_norm.std():.4f}  (Target ~1)")

    # --- START RECONSTRUCTION (Your Pipeline) ---
    print("\n🚀 Starting reconstruction (Denorm -> View -> Decode)...")

    # 1. Denormalize
    latents_recon = (lat_norm * GLOBAL_STD) + GLOBAL_MEAN

    # 2. Shape fix & view
    if latents_recon.shape[-1] == 752: latents_recon = latents_recon[..., :750]

    latents_recon = latents_recon.contiguous()
    input_embeddings = latents_recon.view(b, 128, -1)

    # 3. Decoding
    with torch.no_grad():
        decoded = encodec_model.decoder(input_embeddings)
        audio_out = decoded.squeeze().cpu().numpy()

    if audio_out.ndim > 1: audio_out = audio_out[0]

    # 4. Resample
    audio_16k = librosa.resample(audio_out, orig_sr=24000, target_sr=16000)
    audio_final = audio_16k / (np.max(np.abs(audio_16k)) + 1e-9)

    # --- OUTPUT ---
    print("🎉 PLAYBACK: This is the ORIGINAL (processed through the pipeline).")
    ipd.display(ipd.Audio(audio_final, rate=16000))

    # --- PLOTTING ---
    plt.figure(figsize=(18, 7))

    # LEFT: Spectrogram
    plt.subplot(1, 2, 1)
    D = librosa.amplitude_to_db(np.abs(librosa.stft(audio_final, n_fft=2048)), ref=np.max)
    img = librosa.display.specshow(D, sr=16000, x_axis='time', y_axis='hz', cmap='magma', vmin=-80)
    plt.ylim(0, 4000)
    plt.colorbar(img, format='%+2.0f dB')

    # Target lines
    target_freq = 366.67
    plt.axhline(y=target_freq, color='cyan', linestyle='-', linewidth=2, label='366 Hz')
    plt.axhline(y=target_freq*2, color='cyan', linestyle=':', label='Harmonics')
    plt.legend(loc='upper right')
    plt.title("Spectrogram (Ground Truth Pipeline)")

    # RIGHT: PSD (Energy)
    plt.subplot(1, 2, 2)
    freqs, psd = signal.welch(audio_final, 16000, nperseg=4096)

    # Plotting in GREEN, representing ground truth
    plt.semilogy(freqs, psd, color='#2ecc71', linewidth=1.5, label='Original')

    # Target lines
    plt.axvline(x=target_freq, color='cyan', linestyle='--', linewidth=1.5, label='Target 366 Hz')
    plt.axvline(x=target_freq*2, color='cyan', linestyle=':', alpha=0.6)

    plt.xlim(0, 4000)
    plt.ylim(bottom=1e-7)
    plt.grid(True, which="both", alpha=0.3)
    plt.xlabel('Frequency [Hz]')
    plt.title("PSD Energy Distribution (Must show peaks!)")
    plt.legend()

    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"❌ Error: {e}")

In [ ]:
# ==============================================================================
# STEP 3: MODEL (EPOCH 700) & DDIM SCHEDULER
# ==============================================================================
import os
import glob
import torch
from diffusers import UNet2DConditionModel, DDIMScheduler, DPMSolverMultistepScheduler

# 1. PATHS
BASE_PATH = '/content/drive/MyDrive/MasterProject'
CHECKPOINT_DIR = os.path.join(BASE_PATH, 'outputs', 'results', 'training_logs', 'checkpoints', 'MIMII_Gen_WideUNet_Drop10_20260126_2241')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. ARCHITECTURE (The skeleton)
config = {
    "sample_size": 8,
    "in_channels": 16,
    "out_channels": 16,
    "layers_per_block": 2,
    "norm_num_groups": 16,
    "block_out_channels": (256, 512, 512, 1024),
    "down_block_types": ("CrossAttnDownBlock2D", "CrossAttnDownBlock2D", "CrossAttnDownBlock2D", "CrossAttnDownBlock2D"),
    "up_block_types": ("CrossAttnUpBlock2D", "CrossAttnUpBlock2D", "CrossAttnUpBlock2D", "CrossAttnUpBlock2D"),
    "cross_attention_dim": 768,
}
print("🏗️ Building UNet architecture...")
test_unet = UNet2DConditionModel(**config)

# 3. LOAD WEIGHTS (The memory)
WUNSCH_EPOCHE = "700"  # Target epoch
print(f"📂 Searching specifically for epoch {WUNSCH_EPOCHE} in: {CHECKPOINT_DIR}")

# Searching for file containing the epoch number
found_files = glob.glob(os.path.join(CHECKPOINT_DIR, f"*{WUNSCH_EPOCHE}*.bin"))
if not found_files:
    # Fallback for safetensors
    found_files = glob.glob(os.path.join(CHECKPOINT_DIR, f"*{WUNSCH_EPOCHE}*.safetensors"))

if len(found_files) > 0:
    model_path = found_files[0]
    print(f"🎯 FOUND! Loading file: {os.path.basename(model_path)}")

    state_dict = torch.load(model_path, map_location="cpu")
    test_unet.load_state_dict(state_dict, strict=True)
    test_unet.to(device)
    test_unet.eval()
    print(f"✅ Epoch {WUNSCH_EPOCHE} successfully loaded!")
else:
    # List available files if target epoch is missing
    all_files = os.listdir(CHECKPOINT_DIR)
    print(f"❌ Epoch {WUNSCH_EPOCHE} not found! Available files:")
    print(all_files[:5])
    raise FileNotFoundError("Please check the epoch.")

# 4. SCHEDULER (DDIM)
# Using DDIM for stable inference
print("⚙️ Activating DDIM scheduler...")
scheduler = DDIMScheduler(
    num_train_timesteps=1000,      # Must be 1000 (as trained)
    beta_start=0.00085,            # Exactly as in training
    beta_end=0.012,                # Exactly as in training
    beta_schedule="scaled_linear", # IMPORTANT: "scaled_linear"
    clip_sample=False,             # CRUCIAL: Do not clip audio latents
    prediction_type="epsilon"      # Model predicts noise
)

num_inference_steps = 100
scheduler.set_timesteps(num_inference_steps)
print(f"🚀 Ready. Steps: {num_inference_steps}")

In [ ]:
# ==============================================================================
# STEP: RESCUE THE STATISTICS (GLOBAL_MEAN & STD)
# ==============================================================================
import os
import torch

# 1. Ensure paths
BASE_PATH = '/content/drive/MyDrive/MasterProject'
STATS_PATH = os.path.join(BASE_PATH, 'data/features/bearing_global_stats.pt')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"🔍 Searching for stats file here: {STATS_PATH}")

# 2. Load and assign
if os.path.exists(STATS_PATH):
    stats = torch.load(STATS_PATH, map_location='cpu')

    # Ensure shape matches for broadcasting
    # Latents are [Batch, 16, 8, 752], so we need [1, 16, 1, 1]
    GLOBAL_MEAN = stats['mean'].view(1, 16, 1, 1).to(device)
    GLOBAL_STD = stats['std'].view(1, 16, 1, 1).to(device)

    print("✅ Statistics loaded successfully!")

    # 3. Immediate diagnostic check
    std_val = GLOBAL_STD.mean().item()
    mean_val = GLOBAL_MEAN.mean().item()

    print(f"   📊 Average STD:  {std_val:.4f}")
    print(f"   📊 Average Mean: {mean_val:.4f}")

    if abs(std_val - 1.0) > 0.001:
        print("✅ Check passed: STD is not 1.0 (These are real data!)")
    else:
        # STOP HERE! Do not proceed.
        raise ValueError(f"CRITICAL ERROR: STD is {std_val:.4f}. Looks like dummy data!")

else:
    print("❌ ALERT: File 'bearing_global_stats.pt' not found!")
    # STOP HERE AS WELL! Do not use fallback with torch.ones!
    raise FileNotFoundError(f"CRITICAL: Stats file missing at {STATS_PATH}. Inference not possible.")

In [ ]:
import os
import torch
import torchaudio
import librosa
import numpy as np
import matplotlib.pyplot as plt
from encodec import EncodecModel

# 1. SETUP
device = "cuda" if torch.cuda.is_available() else "cpu"
model = EncodecModel.encodec_model_24khz()
model.set_target_bandwidth(24.0)
model.to(device)
model.eval()

# 2. LOAD FILE (Using librosa instead of torchaudio.load)
real_path = "/content/drive/MyDrive/MasterProject/data/splits/train/bearing/section_00_source_train_normal_0000_vel_22.wav"

if not os.path.exists(real_path):
    print("❌ File not found! Please check the path.")
else:
    print(f"📂 Loading file with librosa: {real_path}")
    # Librosa loads reliably and converts directly to 24kHz
    y, _ = librosa.load(real_path, sr=24000)
    # Convert to tensor [Batch, Channels, Samples]
    wav = torch.from_numpy(y).unsqueeze(0).unsqueeze(0).to(device)

    # 3. PURE ENCODEC TEST
    with torch.no_grad():
        # Encoder -> Decoder
        latents = model.encoder(wav)
        decoded = model.decoder(latents)

    # 4. VISUALIZATION
    audio_orig = y
    audio_recon = decoded.squeeze().cpu().numpy()

    plt.figure(figsize=(15, 6))

    # Original
    plt.subplot(1, 2, 1)
    D_orig = librosa.amplitude_to_db(np.abs(librosa.stft(audio_orig)), ref=np.max)
    librosa.display.specshow(D_orig, sr=24000, x_axis='time', y_axis='hz', cmap='magma')
    plt.ylim(0, 1000)
    plt.axhline(y=366.6, color='cyan', linestyle='--', label='366 Hz Target')
    plt.title("ORIGINAL (Real file)")
    plt.legend()

    # EnCodec Reconstruction
    plt.subplot(1, 2, 2)
    D_recon = librosa.amplitude_to_db(np.abs(librosa.stft(audio_recon)), ref=np.max)
    librosa.display.specshow(D_recon, sr=24000, x_axis='time', y_axis='hz', cmap='magma')
    plt.ylim(0, 1000)
    plt.axhline(y=366.6, color='cyan', linestyle='--', label='366 Hz Target')
    plt.title("ENCODEC ONLY (Compression effect)")
    plt.legend()

    plt.tight_layout()
    plt.show()

    # --- THE MOMENT OF TRUTH ---
    print("\n🔎 ANALYSIS FOR YOUR THESIS:")
    print("1. Is the cyan line on the right (EnCodec) significantly weaker than on the left?")
    print("2. Do you see 'smearing' or vertical stripes?")
    print("-" * 30)
    print("RESULT: If the line on the right is missing, EnCodec is your bottleneck.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal
import torch
import librosa

# --- SETUP & LOAD (as before) ---
# ... (Load model and prepare wav) ...

# 1. EnCodec Roundtrip
with torch.no_grad():
    latents = model.encoder(wav)
    decoded = model.decoder(latents)

audio_orig = y
audio_recon = decoded.squeeze().cpu().numpy()
sr = 24000

# 2. PSD CALCULATION (Welch's method)
# nperseg determines the frequency resolution. 2048 is a good value for 24kHz.
f_orig, pxx_orig = scipy.signal.welch(audio_orig, fs=sr, nperseg=2048)
f_recon, pxx_recon = scipy.signal.welch(audio_recon, fs=sr, nperseg=2048)

# 3. VISUALIZATION
plt.figure(figsize=(12, 6))

# Plotting in logarithmic scale for better visibility of the dynamic range
plt.semilogy(f_orig, pxx_orig, label='Original (Empirical)', color='blue', alpha=0.7)
plt.semilogy(f_recon, pxx_recon, label='EnCodec Reconstruction', color='red', linestyle='--', alpha=0.8)

# Mark target frequency (366.6 Hz for 22k RPM)
plt.axvline(x=366.6, color='cyan', linestyle=':', label='366.6 Hz Peak')

plt.title("Power Spectral Density (PSD) Comparison")
plt.xlabel("Frequency [Hz]")
plt.ylabel("Power Spectral Density [V^2/Hz]")
plt.xlim(0, 2000) # Focus on the relevant range up to 2kHz
plt.grid(True, which="both", ls="-", alpha=0.5)
plt.legend()

plt.tight_layout()
plt.show()

# Analysis output
peak_orig = pxx_orig[np.argmin(np.abs(f_orig - 366.6))]
peak_recon = pxx_recon[np.argmin(np.abs(f_recon - 366.6))]
print(f"🔎 Energy check at 366.6 Hz:")
print(f"Original: {peak_orig:.2e} | Reconstructed: {peak_recon:.2e}")
print(f"Preservation rate: {(peak_recon/peak_orig)*100:.2f}%")

In [ ]:
from transformers import T5Tokenizer, T5EncoderModel
import torch

# We use the standard t5-base
t5_model_name = "google/flan-t5-base"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"⏳ Loading {t5_model_name} (Encoder & Tokenizer)...")

# 1. Load tokenizer
# model_max_length=512 is the standard safety margin.
# legacy=False prevents annoying warnings.
tokenizer = T5Tokenizer.from_pretrained(t5_model_name, model_max_length=512, legacy=False)

# 2. Load encoder and move it to the GPU
t5_model = T5EncoderModel.from_pretrained(t5_model_name).to(device)
t5_model.eval() # IMPORTANT: Do not forget this, it saves memory!

print(f"✅ t5-base successfully loaded!")
print(f"📏 Embedding dimension: {t5_model.config.d_model} (Must be 768 to match the UNet)")

In [ ]:
# @title
import os
import torch
import scipy.io.wavfile as wavfile
import soundfile as sf
import numpy as np
import librosa
from tqdm.auto import tqdm

# ==============================================================================
# MASS PRODUCTION: SECTION 00 (BEARING) - 1001 FILES (Source & Target Mix)
# ==============================================================================

# 1. SETUP & PATHS
OUTPUT_DIR = "/content/drive/MyDrive/MasterProject/generated_data/bearing/section_00"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- SAFETY NET: Load EnCodec model ---
if 'encodec_model' not in locals():
    print("⏳ Loading EnCodec model for decoding...")
    from encodec import EncodecModel
    encodec_model = EncodecModel.encodec_model_24khz()
    encodec_model.to(device)
    encodec_model.set_target_bandwidth(24.0)
    encodec_model.eval()
    print("✅ EnCodec ready.")

# Definition of domains according to your table
source_velocities = [6, 10, 14, 18, 22]
target_velocities = [2, 4, 8, 12, 16, 20, 24, 26]
all_velocities = source_velocities + target_velocities

num_per_velocity = 77
guidance_scale = 7.0
num_inference_steps = 100
SCALE_FACTOR = 1.0

# --- FAIL-SAFE: Calculate unconditional embedding for CFG ---
print("⏳ Calculating unconditional embedding...")
with torch.no_grad():
    uncond_inputs = tokenizer("", padding="max_length", max_length=64, truncation=True, return_tensors="pt").to(device)
    uncond_emb = t5_model(uncond_inputs.input_ids).last_hidden_state

print(f"🚀 Starting production for section 00. Target: {len(all_velocities)*num_per_velocity} files.")

# Set scheduler timesteps once (saves time on L4)
scheduler.set_timesteps(num_inference_steps)

# 2. MAIN LOOP OVER VELOCITIES
for vel in all_velocities:
    # Determine domain
    domain = "source" if vel in source_velocities else "target"
    print(f"\n--- 📈 Velocity: {vel} krpm ({domain.upper()}) ---")

    # Prepare prompt
    prompt = f"A bearing operating with anomaly at a rotation velocity of {vel} krpm due to eccentricity. The sound contains factory noise at 12.0 dB SNR."

    with torch.no_grad():
        inputs = tokenizer(prompt, padding="max_length", max_length=64, truncation=True, return_tensors="pt").to(device)
        cond_emb = t5_model(inputs.input_ids).last_hidden_state
        prompt_embeds = torch.cat([uncond_emb, cond_emb])

    # 3. LOOP FOR THE 77 VARIANTS
    for i in range(num_per_velocity):
        current_seed = torch.seed()
        generator = torch.Generator(device=device).manual_seed(current_seed)

        # Initial noise
        latents = torch.randn((1, 16, 8, 752), generator=generator, device=device)
        latents = latents * scheduler.init_noise_sigma

        # C. DIFFUSION LOOP
        for t in tqdm(scheduler.timesteps, desc=f"Vel {vel} | {i+1}/{num_per_velocity}", leave=False):
            latent_model_input = torch.cat([latents] * 2)

            with torch.no_grad():
                noise_pred_full = test_unet(latent_model_input, t, encoder_hidden_states=prompt_embeds).sample

            noise_pred_uncond, noise_pred_text = noise_pred_full.chunk(2)
            noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)
            latents = scheduler.step(noise_pred, t, latents).prev_sample

        # D. DECODING & FIXES
        with torch.no_grad():
            # Denormalization with your Global Stats
            current_lat = (latents * (GLOBAL_STD * SCALE_FACTOR + 1e-6)) + GLOBAL_MEAN

            # Shape fix (752 -> 750) & view for EnCodec
            if current_lat.shape[-1] == 752:
                current_lat = current_lat[..., :750]

            input_embeddings = current_lat.contiguous().view(1, 128, 750)
            generated_waveform = encodec_model.decoder(input_embeddings)
            audio_out = generated_waveform.squeeze().cpu().numpy()

        # E. POST-PROCESSING
        if audio_out.ndim > 1: audio_out = audio_out[0]
        audio_16k = librosa.resample(audio_out, orig_sr=24000, target_sr=16000)
        audio_final = audio_16k / (np.max(np.abs(audio_16k)) + 1e-9)

        # F. SAVING (DCASE-compliant name)
        # Example: section_00_source_train_anomaly_0042_seed_98765_vel_14.wav
        filename = f"section_00_{domain}_train_anomaly_{i:04d}_seed_{current_seed}_vel_{vel}.wav"
        save_path = os.path.join(OUTPUT_DIR, filename)

        sf.write(save_path, audio_final, 16000)

print(f"\n✅ DONE! All files for section 00 have been saved to {OUTPUT_DIR}.")

In [ ]:
# @title
import os
import torch
import scipy.io.wavfile as wavfile
import soundfile as sf
import numpy as np
import librosa
from tqdm.auto import tqdm

# ==============================================================================
# MASS PRODUCTION: SECTION 01 (BEARING) - ~1000 FILES
# ==============================================================================

# 1. SETUP & PATHS
SECTION = "section_01"
OUTPUT_DIR = f"/content/drive/MyDrive/MasterProject/generated_data/bearing/{SECTION}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

# --- Load EnCodec model ---
if 'encodec_model' not in locals():
    print("⏳ Loading EnCodec model...")
    from encodec import EncodecModel
    encodec_model = EncodecModel.encodec_model_24khz()
    encodec_model.to(device)
    encodec_model.set_target_bandwidth(24.0)
    encodec_model.eval()
    print("✅ EnCodec ready.")

# Configuration for Section 01
source_configs = {
    "velocities": [4, 12],
    "locations": ["A", "B", "C", "D"]
}
target_configs = {
    "velocities": [4, 12],
    "locations": ["E", "F", "G", "H"]
}

num_per_combination = 63  # 16 combinations * 63 = 1008 files
guidance_scale = 7.0
num_inference_steps = 100
SCALE_FACTOR = 1.0

# --- Calculate unconditional embedding ---
print("⏳ Calculating unconditional embedding...")
with torch.no_grad():
    uncond_inputs = tokenizer("", padding="max_length", max_length=64, truncation=True, return_tensors="pt").to(device)
    uncond_emb = t5_model(uncond_inputs.input_ids).last_hidden_state

print(f"🚀 Starting production for {SECTION}. Target: ~1000 files.")
scheduler.set_timesteps(num_inference_steps)

# 2. MAIN LOOP OVER DOMAINS
for domain_name, config in [("source", source_configs), ("target", target_configs)]:
    for vel in config["velocities"]:
        for loc in config["locations"]:
            print(f"\n--- 📈 {domain_name.upper()} | Speed: {vel} krpm | Loc: {loc} ---")

            # Prompt adjustment: We use your Section 01 template,
            # but change "normally" to "with anomaly due to eccentricity"
            prompt = f"A bearing operating with anomaly at {vel} krpm recorded at microphone location {loc}. The sound contains factory noise at 12.0 dB SNR."

            with torch.no_grad():
                inputs = tokenizer(prompt, padding="max_length", max_length=64, truncation=True, return_tensors="pt").to(device)
                cond_emb = t5_model(inputs.input_ids).last_hidden_state
                prompt_embeds = torch.cat([uncond_emb, cond_emb])

            # 3. LOOP FOR VARIANTS
            for i in range(num_per_combination):
                current_seed = torch.seed()
                generator = torch.Generator(device=device).manual_seed(current_seed)

                # Diffusion
                latents = torch.randn((1, 16, 8, 752), generator=generator, device=device)
                latents = latents * scheduler.init_noise_sigma

                for t in tqdm(scheduler.timesteps, desc=f"Vel {vel} Loc {loc} | {i+1}/{num_per_combination}", leave=False):
                    latent_model_input = torch.cat([latents] * 2)
                    with torch.no_grad():
                        noise_pred_full = test_unet(latent_model_input, t, encoder_hidden_states=prompt_embeds).sample

                    noise_pred_uncond, noise_pred_text = noise_pred_full.chunk(2)
                    noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)
                    latents = scheduler.step(noise_pred, t, latents).prev_sample

                # Decoding
                with torch.no_grad():
                    current_lat = (latents * (GLOBAL_STD * SCALE_FACTOR + 1e-6)) + GLOBAL_MEAN
                    if current_lat.shape[-1] == 752:
                        current_lat = current_lat[..., :750]
                    input_embeddings = current_lat.contiguous().view(1, 128, 750)
                    generated_waveform = encodec_model.decoder(input_embeddings)
                    audio_out = generated_waveform.squeeze().cpu().numpy()

                # Post-Processing
                if audio_out.ndim > 1: audio_out = audio_out[0]
                audio_16k = librosa.resample(audio_out, orig_sr=24000, target_sr=16000)
                audio_final = audio_16k / (np.max(np.abs(audio_16k)) + 1e-9)

                # Saving (DCASE-compliant)
                # Example: section_01_source_train_anomaly_0042_seed_98765_vel_4_loc_A.wav
                filename = f"{SECTION}_{domain_name}_train_anomaly_{i:04d}_seed_{current_seed}_vel_{vel}_loc_{loc}.wav"
                save_path = os.path.join(OUTPUT_DIR, filename)
                sf.write(save_path, audio_final, 16000)

print(f"\n✅ DONE! Section 01 saved to {OUTPUT_DIR}")

In [ ]:
# @title
import os
import torch
import scipy.io.wavfile as wavfile
import soundfile as sf
import numpy as np
import librosa
from tqdm.auto import tqdm

# ==============================================================================
# MASS PRODUCTION: SECTION 02 (BEARING) - ~1000 FILES
# ==============================================================================

# 1. SETUP & PATHS
SECTION = "section_02"
OUTPUT_DIR = f"/content/drive/MyDrive/MasterProject/generated_data/bearing/{SECTION}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

# --- Load EnCodec model ---
if 'encodec_model' not in locals():
    print("⏳ Loading EnCodec model for decoding...")
    from encodec import EncodecModel
    encodec_model = EncodecModel.encodec_model_24khz()
    encodec_model.to(device)
    encodec_model.set_target_bandwidth(24.0)
    encodec_model.eval()
    print("✅ EnCodec ready.")

# Configuration for Section 02
# Source: 6 & 14 krpm with Factory Noise A & B
source_configs = {
    "velocities": [6, 14],
    "noise_types": ["A", "B"]
}
# Target: 6 & 14 krpm with Factory Noise C
target_configs = {
    "velocities": [6, 14],
    "noise_types": ["C"]
}

# 6 combinations * 167 = 1002 files
num_per_combination = 167
guidance_scale = 7.0
num_inference_steps = 100
SCALE_FACTOR = 1.0

# --- Calculate unconditional embedding ---
print("⏳ Calculating unconditional embedding...")
with torch.no_grad():
    uncond_inputs = tokenizer("", padding="max_length", max_length=64, truncation=True, return_tensors="pt").to(device)
    uncond_emb = t5_model(uncond_inputs.input_ids).last_hidden_state

print(f"🚀 Starting production for {SECTION}. Target: ~1000 files.")
scheduler.set_timesteps(num_inference_steps)

# 2. MAIN LOOP OVER DOMAINS
for domain_name, config in [("source", source_configs), ("target", target_configs)]:
    for vel in config["velocities"]:
        for n_type in config["noise_types"]:
            print(f"\n--- 📈 {domain_name.upper()} | Speed: {vel} krpm | Noise: {n_type} ---")

            # Prompt adjustment: "normally" to "with anomaly due to eccentricity"
            # Including factory noise type specification
            prompt = f"A bearing operating with anomaly at {vel} krpm due to eccentricity with background factory noise type {n_type}. The sound contains factory noise at 12.0 dB SNR."

            with torch.no_grad():
                inputs = tokenizer(prompt, padding="max_length", max_length=64, truncation=True, return_tensors="pt").to(device)
                cond_emb = t5_model(inputs.input_ids).last_hidden_state
                prompt_embeds = torch.cat([uncond_emb, cond_emb])

            # 3. LOOP FOR THE 167 VARIANTS
            for i in range(num_per_combination):
                current_seed = torch.seed()
                generator = torch.Generator(device=device).manual_seed(current_seed)

                # Diffusion
                latents = torch.randn((1, 16, 8, 752), generator=generator, device=device)
                latents = latents * scheduler.init_noise_sigma

                for t in tqdm(scheduler.timesteps, desc=f"Vel {vel} Noise {n_type} | {i+1}/{num_per_combination}", leave=False):
                    latent_model_input = torch.cat([latents] * 2)
                    with torch.no_grad():
                        noise_pred_full = test_unet(latent_model_input, t, encoder_hidden_states=prompt_embeds).sample

                    noise_pred_uncond, noise_pred_text = noise_pred_full.chunk(2)
                    noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)
                    latents = scheduler.step(noise_pred, t, latents).prev_sample

                # Decoding & Fixes
                with torch.no_grad():
                    current_lat = (latents * (GLOBAL_STD * SCALE_FACTOR + 1e-6)) + GLOBAL_MEAN
                    if current_lat.shape[-1] == 752:
                        current_lat = current_lat[..., :750]

                    input_embeddings = current_lat.contiguous().view(1, 128, 750)
                    generated_waveform = encodec_model.decoder(input_embeddings)
                    audio_out = generated_waveform.squeeze().cpu().numpy()

                # Post-Processing
                if audio_out.ndim > 1: audio_out = audio_out[0]
                audio_16k = librosa.resample(audio_out, orig_sr=24000, target_sr=16000)
                audio_final = audio_16k / (np.max(np.abs(audio_16k)) + 1e-9)

                # Saving (DCASE-compliant for Section 02)
                # Example: section_02_source_train_anomaly_0042_seed_..._vel_6_f-n_A.wav
                filename = f"{SECTION}_{domain_name}_train_anomaly_{i:04d}_seed_{current_seed}_vel_{vel}_f-n_{n_type}.wav"
                save_path = os.path.join(OUTPUT_DIR, filename)

                sf.write(save_path, audio_final, 16000)

print(f"\n✅ DONE! Section 02 saved to {OUTPUT_DIR}")

In [ ]:
# @title
import os
import torch
import scipy.io.wavfile as wavfile
import soundfile as sf
import numpy as np
import librosa
from tqdm.auto import tqdm

# ==============================================================================
# MASS PRODUCTION: SECTION 03 (BEARING) - ~1000 FILES
# ==============================================================================

# 1. SETUP & PATHS
SECTION = "section_03"
OUTPUT_DIR = f"/content/drive/MyDrive/MasterProject/generated_data/bearing/{SECTION}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

# --- Load EnCodec model ---
if 'encodec_model' not in locals():
    print("⏳ Loading EnCodec model for decoding...")
    from encodec import EncodecModel
    encodec_model = EncodecModel.encodec_model_24khz()
    encodec_model.to(device)
    encodec_model.set_target_bandwidth(24.0)
    encodec_model.eval()
    print("✅ EnCodec ready.")

# Configuration for Section 03 according to your list
source_velocities = [5, 9, 13, 17, 21]
target_velocities = [7, 11, 15, 19]
all_velocities = source_velocities + target_velocities

# 9 velocities * 111 = 999 files
num_per_velocity = 111
guidance_scale = 7.0
num_inference_steps = 100
SCALE_FACTOR = 1.0

# --- Calculate unconditional embedding ---
print("⏳ Calculating unconditional embedding...")
with torch.no_grad():
    uncond_inputs = tokenizer("", padding="max_length", max_length=64, truncation=True, return_tensors="pt").to(device)
    uncond_emb = t5_model(uncond_inputs.input_ids).last_hidden_state

print(f"🚀 Starting production for {SECTION}. Target: ~1000 files.")
scheduler.set_timesteps(num_inference_steps)

# 2. MAIN LOOP OVER VELOCITIES
for vel in all_velocities:
    # Determine domain
    domain = "source" if vel in source_velocities else "target"
    print(f"\n--- 📈 Velocity: {vel} krpm ({domain.upper()}) ---")

    # Prompt adjustment for Section 03 (Anomaly focus)
    prompt = f"A bearing operating with anomaly at a rotation velocity of {vel} krpm due to eccentricity. The sound contains factory noise at 12.0 dB SNR."

    with torch.no_grad():
        inputs = tokenizer(prompt, padding="max_length", max_length=64, truncation=True, return_tensors="pt").to(device)
        cond_emb = t5_model(inputs.input_ids).last_hidden_state
        prompt_embeds = torch.cat([uncond_emb, cond_emb])

    # 3. LOOP FOR THE VARIANTS
    for i in range(num_per_velocity):
        current_seed = torch.seed()
        generator = torch.Generator(device=device).manual_seed(current_seed)

        # Initial noise
        latents = torch.randn((1, 16, 8, 752), generator=generator, device=device)
        latents = latents * scheduler.init_noise_sigma

        # DIFFUSION LOOP
        for t in tqdm(scheduler.timesteps, desc=f"Vel {vel} | {i+1}/{num_per_velocity}", leave=False):
            latent_model_input = torch.cat([latents] * 2)

            with torch.no_grad():
                noise_pred_full = test_unet(latent_model_input, t, encoder_hidden_states=prompt_embeds).sample

            noise_pred_uncond, noise_pred_text = noise_pred_full.chunk(2)
            noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)
            latents = scheduler.step(noise_pred, t, latents).prev_sample

        # DECODING & FIXES
        with torch.no_grad():
            current_lat = (latents * (GLOBAL_STD * SCALE_FACTOR + 1e-6)) + GLOBAL_MEAN

            if current_lat.shape[-1] == 752:
                current_lat = current_lat[..., :750]

            input_embeddings = current_lat.contiguous().view(1, 128, 750)
            generated_waveform = encodec_model.decoder(input_embeddings)
            audio_out = generated_waveform.squeeze().cpu().numpy()

        # POST-PROCESSING
        if audio_out.ndim > 1: audio_out = audio_out[0]
        audio_16k = librosa.resample(audio_out, orig_sr=24000, target_sr=16000)
        audio_final = audio_16k / (np.max(np.abs(audio_16k)) + 1e-9)

        # SAVING
        # Example: section_03_source_train_anomaly_0042_seed_..._vel_9.wav
        filename = f"{SECTION}_{domain}_train_anomaly_{i:04d}_seed_{current_seed}_vel_{vel}.wav"
        save_path = os.path.join(OUTPUT_DIR, filename)

        sf.write(save_path, audio_final, 16000)

print(f"\n✅ DONE! Section 03 saved to {OUTPUT_DIR}.")

In [ ]:
# @title
import os
import torch
import scipy.io.wavfile as wavfile
import soundfile as sf
import numpy as np
import librosa
from tqdm.auto import tqdm

# ==============================================================================
# MASS PRODUCTION: SECTION 04 (BEARING) - ~1000 FILES
# ==============================================================================

# 1. SETUP & PATHS
SECTION = "section_04"
OUTPUT_DIR = f"/content/drive/MyDrive/MasterProject/generated_data/bearing/{SECTION}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

# --- Load EnCodec model ---
if 'encodec_model' not in locals():
    print("⏳ Loading EnCodec model for decoding...")
    from encodec import EncodecModel
    encodec_model = EncodecModel.encodec_model_24khz()
    encodec_model.to(device)
    encodec_model.set_target_bandwidth(24.0)
    encodec_model.eval()
    print("✅ EnCodec ready.")

# Configuration for Section 04 (According to metadata)
source_configs = {
    "velocities": [8, 16],
    "locations": ["A", "B", "C", "D"]
}
target_configs = {
    "velocities": [8, 16],
    "locations": ["E", "F", "G", "H"]
}

# 16 combinations * 63 = 1008 files
num_per_combination = 63
guidance_scale = 7.0
num_inference_steps = 100
SCALE_FACTOR = 1.0

# --- Calculate unconditional embedding ---
print("⏳ Calculating unconditional embedding...")
with torch.no_grad():
    uncond_inputs = tokenizer("", padding="max_length", max_length=64, truncation=True, return_tensors="pt").to(device)
    uncond_emb = t5_model(uncond_inputs.input_ids).last_hidden_state

print(f"🚀 Starting production for {SECTION}. Target: ~1000 files.")
scheduler.set_timesteps(num_inference_steps)

# 2. MAIN LOOP OVER DOMAINS
for domain_name, config in [("source", source_configs), ("target", target_configs)]:
    for vel in config["velocities"]:
        for loc in config["locations"]:
            print(f"\n--- 📈 {domain_name.upper()} | Speed: {vel} krpm | Loc: {loc} ---")

            # Prompt adjustment: Focus on anomaly/eccentricity
            prompt = f"A bearing operating with anomaly at {vel} krpm recorded at microphone location {loc}. The sound contains factory noise at 12.0 dB SNR."

            with torch.no_grad():
                inputs = tokenizer(prompt, padding="max_length", max_length=64, truncation=True, return_tensors="pt").to(device)
                cond_emb = t5_model(inputs.input_ids).last_hidden_state
                prompt_embeds = torch.cat([uncond_emb, cond_emb])

            # 3. LOOP FOR THE VARIANTS
            for i in range(num_per_combination):
                current_seed = torch.seed()
                generator = torch.Generator(device=device).manual_seed(current_seed)

                # Diffusion initialization
                latents = torch.randn((1, 16, 8, 752), generator=generator, device=device)
                latents = latents * scheduler.init_noise_sigma

                # Inference loop (100 steps)
                for t in tqdm(scheduler.timesteps, desc=f"Vel {vel} Loc {loc} | {i+1}/{num_per_combination}", leave=False):
                    latent_model_input = torch.cat([latents] * 2)
                    with torch.no_grad():
                        noise_pred_full = test_unet(latent_model_input, t, encoder_hidden_states=prompt_embeds).sample

                    noise_pred_uncond, noise_pred_text = noise_pred_full.chunk(2)
                    noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)
                    latents = scheduler.step(noise_pred, t, latents).prev_sample

                # Decoding
                with torch.no_grad():
                    current_lat = (latents * (GLOBAL_STD * SCALE_FACTOR + 1e-6)) + GLOBAL_MEAN
                    if current_lat.shape[-1] == 752:
                        current_lat = current_lat[..., :750]

                    input_embeddings = current_lat.contiguous().view(1, 128, 750)
                    generated_waveform = encodec_model.decoder(input_embeddings)
                    audio_out = generated_waveform.squeeze().cpu().numpy()

                # Post-processing (16kHz resampling & normalization)
                if audio_out.ndim > 1: audio_out = audio_out[0]
                audio_16k = librosa.resample(audio_out, orig_sr=24000, target_sr=16000)
                audio_final = audio_16k / (np.max(np.abs(audio_16k)) + 1e-9)

                # Saving (DCASE-compliant)
                filename = f"{SECTION}_{domain_name}_train_anomaly_{i:04d}_seed_{current_seed}_vel_{vel}_loc_{loc}.wav"
                save_path = os.path.join(OUTPUT_DIR, filename)
                sf.write(save_path, audio_final, 16000)

print(f"\n✅ DONE! Section 04 saved to {OUTPUT_DIR}")

# Automatic shutdown command for Windows (local)
# os.system("shutdown /s /t 60")

In [ ]:
# @title
import os
import torch
import scipy.io.wavfile as wavfile
import soundfile as sf
import numpy as np
import librosa
from tqdm.auto import tqdm

# ==============================================================================
# MASS PRODUCTION: SECTION 05 (BEARING) - ~1000 FILES
# ==============================================================================

# 1. SETUP & PATHS
SECTION = "section_05"
OUTPUT_DIR = f"/content/drive/MyDrive/MasterProject/generated_data/bearing/{SECTION}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

# --- Load EnCodec model ---
if 'encodec_model' not in locals():
    print("⏳ Loading EnCodec model for decoding...")
    from encodec import EncodecModel
    encodec_model = EncodecModel.encodec_model_24khz()
    encodec_model.to(device)
    encodec_model.set_target_bandwidth(24.0)
    encodec_model.eval()
    print("✅ EnCodec ready.")

# Configuration for Section 05 (According to metadata)
# Source: 2 & 10 krpm with Factory Noise A & B
source_configs = {
    "velocities": [2, 10],
    "noise_types": ["A", "B"]
}
# Target: 2 & 10 krpm with Factory Noise D
target_configs = {
    "velocities": [2, 10],
    "noise_types": ["D"]
}

# 6 combinations * 167 = 1002 files
num_per_combination = 167
guidance_scale = 7.0
num_inference_steps = 100
SCALE_FACTOR = 1.0

# --- Calculate unconditional embedding ---
print("⏳ Calculating unconditional embedding...")
with torch.no_grad():
    uncond_inputs = tokenizer("", padding="max_length", max_length=64, truncation=True, return_tensors="pt").to(device)
    uncond_emb = t5_model(uncond_inputs.input_ids).last_hidden_state

print(f"🚀 Starting production for {SECTION}. Target: ~1000 files.")
scheduler.set_timesteps(num_inference_steps)

# 2. MAIN LOOP OVER DOMAINS
for domain_name, config in [("source", source_configs), ("target", target_configs)]:
    for vel in config["velocities"]:
        for n_type in config["noise_types"]:
            print(f"\n--- 📈 {domain_name.upper()} | Speed: {vel} krpm | Noise: {n_type} ---")

            # Prompt adjustment: Focus on eccentricity anomaly and noise type
            prompt = f"A bearing operating with anomaly at {vel} krpm due to eccentricity with background factory noise type {n_type}. The sound contains factory noise at 12.0 dB SNR."

            with torch.no_grad():
                inputs = tokenizer(prompt, padding="max_length", max_length=64, truncation=True, return_tensors="pt").to(device)
                cond_emb = t5_model(inputs.input_ids).last_hidden_state
                prompt_embeds = torch.cat([uncond_emb, cond_emb])

            # 3. LOOP FOR THE 167 VARIANTS
            for i in range(num_per_combination):
                current_seed = torch.seed()
                generator = torch.Generator(device=device).manual_seed(current_seed)

                # Diffusion initialization
                latents = torch.randn((1, 16, 8, 752), generator=generator, device=device)
                latents = latents * scheduler.init_noise_sigma

                # Inference loop
                for t in tqdm(scheduler.timesteps, desc=f"Vel {vel} Noise {n_type} | {i+1}/{num_per_combination}", leave=False):
                    latent_model_input = torch.cat([latents] * 2)
                    with torch.no_grad():
                        noise_pred_full = test_unet(latent_model_input, t, encoder_hidden_states=prompt_embeds).sample

                    noise_pred_uncond, noise_pred_text = noise_pred_full.chunk(2)
                    noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)
                    latents = scheduler.step(noise_pred, t, latents).prev_sample

                # Decoding & scaling
                with torch.no_grad():
                    current_lat = (latents * (GLOBAL_STD * SCALE_FACTOR + 1e-6)) + GLOBAL_MEAN
                    if current_lat.shape[-1] == 752:
                        current_lat = current_lat[..., :750]

                    input_embeddings = current_lat.contiguous().view(1, 128, 750)
                    generated_waveform = encodec_model.decoder(input_embeddings)
                    audio_out = generated_waveform.squeeze().cpu().numpy()

                # Post-processing
                if audio_out.ndim > 1: audio_out = audio_out[0]
                audio_16k = librosa.resample(audio_out, orig_sr=24000, target_sr=16000)
                audio_final = audio_16k / (np.max(np.abs(audio_16k)) + 1e-9)

                # Saving (DCASE-compliant for Section 05)
                # Example: section_05_source_train_anomaly_0042_seed_..._vel_2_f-n_B.wav
                filename = f"{SECTION}_{domain_name}_train_anomaly_{i:04d}_seed_{current_seed}_vel_{vel}_f-n_{n_type}.wav"
                save_path = os.path.join(OUTPUT_DIR, filename)

                sf.write(save_path, audio_final, 16000)

print(f"\n✅ DONE! Section 05 saved to {OUTPUT_DIR}.")

In [ ]:
# 1. The real target (Conditional)
prompt = "A bearing operating normally at a rotation velocity of 22 krpm. The sound contains factory noise at 12.0 dB SNR."
#prompt = "A bearing operating with anomaly at a rotation velocity of 22 krpm due to eccentricity. The sound contains factory noise at 12.0 dB SNR."
inputs = tokenizer(prompt, padding="max_length", max_length=64, truncation=True, return_tensors="pt").to(device)

# 2. The empty target (Unconditional / Negative example)
# This is the key for CFG!
uncond_inputs = tokenizer("", padding="max_length", max_length=64, truncation=True, return_tensors="pt").to(device)

with torch.no_grad():
    encoder_hidden_states = t5_model(inputs.input_ids).last_hidden_state
    uncond_emb = t5_model(uncond_inputs.input_ids).last_hidden_state

print("✅ Both embeddings (Text & Empty) are ready.")

In [ ]:
# ==============================================================================
# STEP: CREATE INITIAL NOISE (LATENTS)
# ==============================================================================

# 1. Set seed
SEED = 42
generator = torch.Generator(device=device).manual_seed(SEED)

# 2. Create noise
# Shape: [Batch, Channels, Height (Freq), Width (Time)]
latents = torch.randn(
    (1, 16, 8, 752),
    generator=generator,
    device=device
)

# 3. IMPORTANT SCALING (Makes the code compatible with all schedulers)
# For DDIM this is 1.0 (neutral), for DPM++ it is important.
latents = latents * scheduler.init_noise_sigma

print(f"✅ Initial noise created! Shape: {latents.shape}")
print(f"   Sigma scaling was: {scheduler.init_noise_sigma:.4f}")

In [ ]:
# ==============================================================================
# STEP 4: GENERATION (DDPM)
# ==============================================================================
from tqdm.auto import tqdm
import torch

# 1. SETUP
scheduler.set_timesteps(num_inference_steps)
guidance_scale = 7.0
# 7.0 with seed 42 is good
print(f"🌀 Starting DDPM denoising (Steps: {num_inference_steps}, Scale: {guidance_scale})...")

# IMPORTANT: We do NOT create new noise here.
# We use the 'latents' from the previous step.
# We clone it (.clone()) to preserve the original for further tests.
current_latents = latents.clone()

# 2. THE LOOP
for t in tqdm(scheduler.timesteps):
    # A. Duplicate for CFG
    latent_model_input = torch.cat([current_latents] * 2)

    # B. Scaling (DDPM usually doesn't need this, but it doesn't hurt)
    # latent_model_input = scheduler.scale_model_input(latent_model_input, t)

    # C. Embeddings
    prompt_embeds = torch.cat([uncond_emb, encoder_hidden_states])

    # D. Prediction
    with torch.no_grad():
        noise_pred_full = test_unet(
            latent_model_input,
            t,
            encoder_hidden_states=prompt_embeds
        ).sample

    # E. CFG Calculation
    noise_pred_uncond, noise_pred_text = noise_pred_full.chunk(2)
    noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

    # F. Step (Note: we use 'current_latents')
    current_latents = scheduler.step(noise_pred, t, current_latents).prev_sample

# Save result for the next step
latents_final = current_latents
print("✅ Generation finished.")